# Insider Trading Anomaly Detection
**Data:** SEBI PIT (Prohibition of Insider Trading) Annual Disclosures — NSE India  
**Approach:** Unsupervised anomaly detection (Isolation Forest + Autoencoder)  
**Goal:** Flag statistically unusual insider trades for further investigation

---
## Step 1: Load and Clean the Raw Data

### Cell 1 — Imports
Standard libraries we'll use throughout the project:
- `pandas` — for loading and manipulating tabular data
- `numpy` — for numeric operations
- `warnings` — to suppress noisy but harmless pandas warnings

In [4]:
!pip install pandas numpy

     |████████████████████████████████| 10.8 MB 4.2 MB/s eta 0:00:01
     |████████████████████████████████| 5.3 MB 10.3 MB/s eta 0:00:01
     |████████████████████████████████| 349 kB 33.7 MB/s eta 0:00:01
     |████████████████████████████████| 510 kB 19.3 MB/s eta 0:00:01
You should consider upgrading via the '/Users/siddanthsatish/Desktop/ML predictor/.venv/bin/python3 -m pip install --upgrade pip' command.


In [12]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully')

Libraries loaded successfully


### Cell 2 — Load the raw CSV
Two things to note about how we load this file:
- `encoding='utf-8-sig'` strips a hidden BOM byte that NSE's export adds — without this, the first column name gets a garbled prefix
- `low_memory=False` tells pandas not to guess column types mid-file (the mixed `-` and numeric values would confuse it)

We print shape and first 3 rows to confirm it loaded correctly.

In [13]:
RAW_PATH = 'CF-PIT-Annual-equities-02-01-2023-to-01-01-2026.csv'

df = pd.read_csv(RAW_PATH, encoding='utf-8-sig', low_memory=False)

print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head(3)

Shape: 210,392 rows × 19 columns


,COMPANY NAME \n,NAME OF PERSON \n,CATEGORY OF PERSON \n,TYPE \n,SECURITIES ACQUIRED / DISPOSED - BUY \nNo. of shares,SECURITIES ACQUIRED / DISPOSED - BUY \n% of total equity share capital,SECURITIES ACQUIRED / DISPOSED - BUY \nValue of Securities,SECURITIES ACQUIRED / DISPOSED - SALE / PLEDGE INVOCATION \nNo. of shares,SECURITIES ACQUIRED / DISPOSED - SALE / PLEDGE INVOCATION \n% of total equity share capital,SECURITIES ACQUIRED / DISPOSED - SALE / PLEDGE INVOCATION \nValue of Securities,SECURITIES ACQUIRED / DISPOSED - PLEDGE CREATION / PLEDGE RELEASE \nNo. of shares,SECURITIES ACQUIRED / DISPOSED - PLEDGE CREATION / PLEDGE RELEASE \n% of total equity share capital,SECURITIES ACQUIRED / DISPOSED - PLEDGE CREATION / PLEDGE RELEASE \nValue of Securities,SECURITIES HELD POST ACQUISITION / DISPOSAL \nNo. of shares,SECURITIES HELD POST ACQUISITION / DISPOSAL \n% of total equity share capital,TYPE OF CONTRACT \n,CONTRACT SPECIFICATION \n,DATE OF ALLOTMENT ADVICE/ACQUISITION OF SHARES/SALE OF SHARES SPECIFY \nFrom,DATE OF ALLOTMENT ADVICE/ACQUISITION OF SHARES/SALE OF SHARES SPECIFY \nTo
0,Aditya Birla Sun Life AMC Limited,Aditya Birla Money Limited,Designated Person,Equity,-,-,-,-,-,-,"14,774",0,"1,24,60,392",-,-,NaN,NaN,01-Jan-2026,01-Jan-2026
1,Vinsys IT Services India Limited,Karan Vikrant Patil,Promoter,Equity,-,-,-,"14,000",0,"49,63,000",-,-,-,"5,99,000",4,NaN,NaN,01-Jan-2026,01-Jan-2026
2,UTI Asset Management Company Limited,MR. SIDDHARTHA DASH,Designated Person,Equity,"1,620",0,"18,06,786",-,-,-,-,-,-,"4,251",0,NaN,NaN,01-Jan-2026,01-Jan-2026


### Cell 3 — Fix column names
The NSE CSV has multi-line headers (e.g. `SECURITIES ACQUIRED / DISPOSED - BUY \nNo. of shares`).  
Pandas reads the whole thing as one long string with `\n` in the middle.  
We strip those newlines and whitespace, then rename everything to short snake_case names so they're easy to type and reference later.

In [14]:
# Strip embedded newlines — use split/join to also collapse double-spaces
df.columns = [' '.join(c.replace('\n', ' ').split()) for c in df.columns]

# Rename to short, clean snake_case names
COLUMN_MAP = {
    'COMPANY NAME': 'company',
    'NAME OF PERSON': 'person',
    'CATEGORY OF PERSON': 'category',
    'TYPE': 'security_type',
    'SECURITIES ACQUIRED / DISPOSED - BUY No. of shares': 'buy_shares',
    'SECURITIES ACQUIRED / DISPOSED - BUY % of total equity share capital': 'buy_pct',
    'SECURITIES ACQUIRED / DISPOSED - BUY Value of Securities': 'buy_value',
    'SECURITIES ACQUIRED / DISPOSED - SALE / PLEDGE INVOCATION No. of shares': 'sale_shares',
    'SECURITIES ACQUIRED / DISPOSED - SALE / PLEDGE INVOCATION % of total equity share capital': 'sale_pct',
    'SECURITIES ACQUIRED / DISPOSED - SALE / PLEDGE INVOCATION Value of Securities': 'sale_value',
    'SECURITIES ACQUIRED / DISPOSED - PLEDGE CREATION / PLEDGE RELEASE No. of shares': 'pledge_shares',
    'SECURITIES ACQUIRED / DISPOSED - PLEDGE CREATION / PLEDGE RELEASE % of total equity share capital': 'pledge_pct',
    'SECURITIES ACQUIRED / DISPOSED - PLEDGE CREATION / PLEDGE RELEASE Value of Securities': 'pledge_value',
    'SECURITIES HELD POST ACQUISITION / DISPOSAL No. of shares': 'held_shares',
    'SECURITIES HELD POST ACQUISITION / DISPOSAL % of total equity share capital': 'held_pct',
    'TYPE OF CONTRACT': 'contract_type',
    'CONTRACT SPECIFICATION': 'contract_spec',
    'DATE OF ALLOTMENT ADVICE/ACQUISITION OF SHARES/SALE OF SHARES SPECIFY From': 'date_from',
    'DATE OF ALLOTMENT ADVICE/ACQUISITION OF SHARES/SALE OF SHARES SPECIFY To': 'date_to',
}

df = df.rename(columns=COLUMN_MAP)

print('Columns after rename:')
print(df.columns.tolist())
df.head(3)

Columns after rename:
['company', 'person', 'category', 'security_type', 'buy_shares', 'buy_pct', 'buy_value', 'sale_shares', 'sale_pct', 'sale_value', 'pledge_shares', 'pledge_pct', 'pledge_value', 'held_shares', 'held_pct', 'contract_type', 'contract_spec', 'date_from', 'date_to']


,company,person,category,security_type,buy_shares,buy_pct,buy_value,sale_shares,sale_pct,sale_value,pledge_shares,pledge_pct,pledge_value,held_shares,held_pct,contract_type,contract_spec,date_from,date_to
0,Aditya Birla Sun Life AMC Limited,Aditya Birla Money Limited,Designated Person,Equity,-,-,-,-,-,-,"14,774",0,"1,24,60,392",-,-,NaN,NaN,01-Jan-2026,01-Jan-2026
1,Vinsys IT Services India Limited,Karan Vikrant Patil,Promoter,Equity,-,-,-,"14,000",0,"49,63,000",-,-,-,"5,99,000",4,NaN,NaN,01-Jan-2026,01-Jan-2026
2,UTI Asset Management Company Limited,MR. SIDDHARTHA DASH,Designated Person,Equity,"1,620",0,"18,06,786",-,-,-,-,-,-,"4,251",0,NaN,NaN,01-Jan-2026,01-Jan-2026


### Cell 4 — Filter to Equity only, drop near-empty columns
The file contains both `Equity` and `Derivative` rows. We drop derivatives because:
- Derivatives (futures/options) have completely different semantics — the "value" column means notional contract value, not actual cash paid
- You can't compare a ₹10L equity buy directly to a ₹10L derivatives position — apples and oranges for anomaly scoring

We also drop `contract_type` and `contract_spec` — these are ~100% blank for equity rows and only apply to derivatives.

In [15]:
print(f"Security type breakdown:\n{df['security_type'].value_counts()}\n")

df = df[df['security_type'] == 'Equity'].copy()
df = df.drop(columns=['security_type', 'contract_type', 'contract_spec'])

print(f'After filtering to Equity: {df.shape[0]:,} rows × {df.shape[1]} columns')

Security type breakdown:
security_type
Equity        206809
Derivative      3575
Debt               8
Name: count, dtype: int64

After filtering to Equity: 206,809 rows × 16 columns


### Cell 5 — Clean numeric columns
Two problems with every numeric column in this dataset:
1. **`"-"` means zero** (NSE uses a dash instead of `0` when nothing happened)
2. **Indian-format commas** — `"1,24,60,392"` is one crore twenty-four lakh sixty thousand, but Python's `float()` chokes on the commas

Fix: replace `"-"` with `"0"`, strip all commas, then convert to float.  
We also strip whitespace from text columns (company/person names sometimes have trailing spaces).

In [16]:
NUMERIC_COLS = [
    'buy_shares', 'buy_pct', 'buy_value',
    'sale_shares', 'sale_pct', 'sale_value',
    'pledge_shares', 'pledge_pct', 'pledge_value',
    'held_shares', 'held_pct',
]

TEXT_COLS = ['company', 'person', 'category']

def clean_numeric(series):
    return (
        series
        .astype(str)
        .str.strip()
        .replace('-', '0')
        .str.replace(',', '', regex=False)
        .replace('', '0')
        .astype(float)
    )

for col in NUMERIC_COLS:
    df[col] = clean_numeric(df[col])

for col in TEXT_COLS:
    df[col] = df[col].astype(str).str.strip()

print('Numeric columns cleaned. Sample values:')
df[NUMERIC_COLS].describe().round(0)

Numeric columns cleaned. Sample values:


,buy_shares,buy_pct,buy_value,sale_shares,sale_pct,sale_value,pledge_shares,pledge_pct,pledge_value,held_shares,held_pct
count,2.068090e+05,206809.0,2.068090e+05,2.068090e+05,206809.0,2.068090e+05,2.068090e+05,206809.0,2.068090e+05,2.068090e+05,206809.0
mean,4.512440e+05,0.0,1.303053e+08,4.458850e+05,0.0,1.251765e+08,2.518350e+05,0.0,4.996256e+07,1.036848e+07,2.0
std,2.324993e+07,2.0,9.281061e+09,2.290434e+07,2.0,8.672862e+09,9.225872e+06,1.0,1.610670e+09,1.352769e+08,8.0
min,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.0
25%,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.0,0.000000e+00,4.330000e+03,0.0
50%,0.000000e+00,0.0,0.000000e+00,5.000000e+02,0.0,6.471850e+05,0.000000e+00,0.0,0.000000e+00,2.350000e+04,0.0
75%,1.689000e+03,0.0,1.427270e+06,4.471000e+03,0.0,2.764080e+06,0.000000e+00,0.0,0.000000e+00,1.864820e+05,0.0
max,5.593278e+09,669.0,2.596097e+12,5.593278e+09,90.0,2.596097e+12,1.302144e+09,135.0,5.277041e+11,7.391004e+09,669.0


### Cell 6 — Parse dates
Dates are in `dd-Mon-yyyy` format (e.g. `01-Jan-2026`).  
We parse both `date_from` and `date_to`, then add a `trade_year` and `trade_month` column — useful later for time-based features and Power BI filtering.  

We also check the date range to confirm the file covers what we expect.

In [17]:
df['date_from'] = pd.to_datetime(df['date_from'], format='%d-%b-%Y', errors='coerce')
df['date_to']   = pd.to_datetime(df['date_to'],   format='%d-%b-%Y', errors='coerce')

df['trade_year']  = df['date_from'].dt.year
df['trade_month'] = df['date_from'].dt.to_period('M').astype(str)

unparsed = df['date_from'].isna().sum()
print(f'Date range: {df["date_from"].min().date()} → {df["date_from"].max().date()}')
print(f'Unparseable date_from rows: {unparsed}')
print(f'\nTrades per year:')
print(df['trade_year'].value_counts().sort_index())

Date range: 2023-01-02 → 2026-01-01
Unparseable date_from rows: 0

Trades per year:
trade_year
2023    60754
2024    73744
2025    72261
2026       50
Name: count, dtype: int64


### Cell 7 — Derive transaction type
Each row in the data represents ONE disclosure, but the disclosure can describe a buy, a sale, a pledge, or some combination.  
We look at which monetary columns are non-zero to assign a `transaction_type` label.

**Known limitation (intentional):**  
- `sale_value > 0` could mean an ordinary sale OR a pledge invocation (loan default) — the column conflates both  
- `pledge_value > 0` could mean a new pledge OR a pledge release — also conflated  

We label these honestly (`sale_or_pledge_invoke`, `pledge_create_or_release`) rather than guessing. This is a real constraint of the SEBI PIT format and is worth mentioning in the portfolio writeup as a sign of data literacy.

In [18]:
def assign_transaction_type(row):
    has_buy    = row['buy_value'] > 0
    has_sale   = row['sale_value'] > 0
    has_pledge = row['pledge_value'] > 0

    if has_buy and not has_sale and not has_pledge:
        return 'buy'
    elif has_sale and not has_buy and not has_pledge:
        return 'sale_or_pledge_invoke'
    elif has_pledge and not has_buy and not has_sale:
        return 'pledge_create_or_release'
    elif has_buy and has_sale:
        return 'mixed_buy_sale'
    elif has_buy and has_pledge:
        return 'mixed_buy_pledge'
    elif has_sale and has_pledge:
        return 'mixed_sale_pledge'
    else:
        return 'no_value_recorded'

df['transaction_type'] = df.apply(assign_transaction_type, axis=1)

print('Transaction type breakdown:')
print(df['transaction_type'].value_counts())

Transaction type breakdown:
transaction_type
sale_or_pledge_invoke       104421
buy                          61834
pledge_create_or_release     27857
mixed_buy_sale                6882
mixed_sale_pledge             3003
mixed_buy_pledge              2793
no_value_recorded               19
Name: count, dtype: int64


### Cell 8 — Final check and save
Quick sanity check before saving:
- Shape and column list
- Null counts (especially dates and key numeric columns)
- A few sample rows

Then we save to `cleaned_trades.csv` — this becomes the input for Step 2 (feature engineering).

In [19]:
print('=== FINAL CLEANED DATASET ===')
print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'\nNull counts:')
print(df.isnull().sum()[df.isnull().sum() > 0])
print(f'\nCategories: {df["category"].unique()}')
print(f'Transaction types: {df["transaction_type"].value_counts().to_dict()}')

CLEAN_PATH = 'cleaned_trades.csv'
df.to_csv(CLEAN_PATH, index=False)
print(f'\nSaved to {CLEAN_PATH}')

df.head(5)

=== FINAL CLEANED DATASET ===
Shape: 206,809 rows × 19 columns

Null counts:
Series([], dtype: int64)

Categories: ['Designated Person' 'Promoter' 'Promoter And Designated Person']
Transaction types: {'sale_or_pledge_invoke': 104421, 'buy': 61834, 'pledge_create_or_release': 27857, 'mixed_buy_sale': 6882, 'mixed_sale_pledge': 3003, 'mixed_buy_pledge': 2793, 'no_value_recorded': 19}

Saved to cleaned_trades.csv


,company,person,category,buy_shares,buy_pct,buy_value,sale_shares,sale_pct,sale_value,pledge_shares,pledge_pct,pledge_value,held_shares,held_pct,date_from,date_to,trade_year,trade_month,transaction_type
0,Aditya Birla Sun Life AMC Limited,Aditya Birla Money Limited,Designated Person,0.0,0.0,0.0,0.0,0.0,0.0,14774.0,0.0,12460392.0,0.0,0.0,2026-01-01,2026-01-01,2026,2026-01,pledge_create_or_release
1,Vinsys IT Services India Limited,Karan Vikrant Patil,Promoter,0.0,0.0,0.0,14000.0,0.0,4963000.0,0.0,0.0,0.0,599000.0,4.0,2026-01-01,2026-01-01,2026,2026-01,sale_or_pledge_invoke
2,UTI Asset Management Company Limited,MR. SIDDHARTHA DASH,Designated Person,1620.0,0.0,1806786.0,0.0,0.0,0.0,0.0,0.0,0.0,4251.0,0.0,2026-01-01,2026-01-01,2026,2026-01,buy
3,UTI Asset Management Company Limited,MR. SACHIN DINESH TRIVEDI,Designated Person,1818.0,0.0,2027615.0,0.0,0.0,0.0,0.0,0.0,0.0,15840.0,0.0,2026-01-01,2026-01-01,2026,2026-01,buy
4,UTI Asset Management Company Limited,MR. PARAG ARUN CHAVAN,Designated Person,977.0,0.0,1089648.0,0.0,0.0,0.0,0.0,0.0,0.0,4347.0,0.0,2026-01-01,2026-01-01,2026,2026-01,buy


---
## Step 2: Feature Engineering

Raw disclosure data tells us *what* happened. Feature engineering turns that into signals a model can learn from — answering questions like:
- Is this trade unusually large *for this specific insider*?
- How long has it been since they last traded?
- Are multiple insiders at the same company trading around the same time?

We'll load the cleaned file from Step 1 and add these features one by one.

### Cell 9 — Load cleaned data

We reload `cleaned_trades.csv` (output of Step 1) so this step is independent.
We also re-parse the date columns since CSV doesn't preserve datetime types.

In [20]:
import pandas as pd
import numpy as np

df = pd.read_csv('cleaned_trades.csv', low_memory=False)
df['date_from'] = pd.to_datetime(df['date_from'])
df['date_to']   = pd.to_datetime(df['date_to'])

print(f'Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head(3)

Loaded: 206,809 rows × 19 columns


,company,person,category,buy_shares,buy_pct,buy_value,sale_shares,sale_pct,sale_value,pledge_shares,pledge_pct,pledge_value,held_shares,held_pct,date_from,date_to,trade_year,trade_month,transaction_type
0,Aditya Birla Sun Life AMC Limited,Aditya Birla Money Limited,Designated Person,0.0,0.0,0.0,0.0,0.0,0.0,14774.0,0.0,12460392.0,0.0,0.0,2026-01-01,2026-01-01,2026,2026-01,pledge_create_or_release
1,Vinsys IT Services India Limited,Karan Vikrant Patil,Promoter,0.0,0.0,0.0,14000.0,0.0,4963000.0,0.0,0.0,0.0,599000.0,4.0,2026-01-01,2026-01-01,2026,2026-01,sale_or_pledge_invoke
2,UTI Asset Management Company Limited,MR. SIDDHARTHA DASH,Designated Person,1620.0,0.0,1806786.0,0.0,0.0,0.0,0.0,0.0,0.0,4251.0,0.0,2026-01-01,2026-01-01,2026,2026-01,buy


### Cell 10 — Compute `trade_value` and `trade_shares` (single number per row)

Right now the ₹ value of a trade is spread across three separate columns (`buy_value`, `sale_value`, `pledge_value`).
For anomaly scoring we need **one number** that represents "how big was this trade?"

Since these columns are mostly mutually exclusive (a single disclosure is usually a buy OR a sale OR a pledge),
we simply sum them — the non-zero one wins.

We do the same for share count.

In [21]:
df['trade_value']  = df['buy_value']  + df['sale_value']  + df['pledge_value']
df['trade_shares'] = df['buy_shares'] + df['sale_shares'] + df['pledge_shares']

print('trade_value summary (₹):')
print(df['trade_value'].describe().apply(lambda x: f'{x:,.0f}'))
print()
print(f'Rows with trade_value = 0: {(df["trade_value"] == 0).sum():,}  ← disclosures with no monetary value recorded')


trade_value summary (₹):
count              206,809
mean           305,444,387
std         17,252,813,857
min                      0
25%              1,349,414
50%              3,029,500
75%             10,528,988
max      5,192,193,897,424
Name: trade_value, dtype: object

Rows with trade_value = 0: 19  ← disclosures with no monetary value recorded


### Cell 11 — Z-score: how unusual is this trade for *this specific insider*?

**What is a z-score?**
It measures how many standard deviations a value is from the *mean of its group*.
- z = 0 → exactly average for this insider
- z = 2 → 2× the standard deviation above their average (notable)
- z = 4+ → extremely unusual for this person → strong anomaly signal

We group by `(person, company)` because the same individual can be an insider at multiple companies,
and we want to compare each trade against that person's history *at that specific company*.

**Edge case:** If an insider has only 1 trade on record, std dev is 0 and z-score is undefined.
We assign z = 0 in that case (we simply have no history to compare against).

In [22]:
def zscore_safe(series):
    """Z-score within a group; returns 0 if std is 0 (only 1 trade)."""
    mean = series.mean()
    std  = series.std()
    if std == 0 or pd.isna(std):
        return pd.Series(0.0, index=series.index)
    return (series - mean) / std

df['trade_value_zscore'] = (
    df.groupby(['person', 'company'])['trade_value']
    .transform(zscore_safe)
)

print('Z-score distribution:')
print(df['trade_value_zscore'].describe().round(2))
print()
print(f'Trades with |z| > 3  (very unusual): {(df["trade_value_zscore"].abs() > 3).sum():,}')
print(f'Trades with |z| > 5  (extreme):      {(df["trade_value_zscore"].abs() > 5).sum():,}')


Z-score distribution:
count    206809.00
mean          0.00
std           0.91
min          -2.83
25%          -0.59
50%          -0.24
75%           0.41
max          26.90
Name: trade_value_zscore, dtype: float64

Trades with |z| > 3  (very unusual): 2,140
Trades with |z| > 5  (extreme):      281


### Cell 12 — Days since last trade

An insider who suddenly trades after a long silence is more interesting than one who trades regularly.
This feature captures the *gap* between consecutive trades for the same person at the same company.

We sort by person+company+date, then compute the difference between each row's date and the previous one.
For the very first trade in a person's history there's no "previous" — we fill that with -1 as a sentinel.

In [23]:
df = df.sort_values(['person', 'company', 'date_from']).reset_index(drop=True)

df['days_since_last_trade'] = (
    df.groupby(['person', 'company'])['date_from']
    .diff()
    .dt.days
    .fillna(-1)   # -1 = first trade ever recorded for this insider
    .astype(int)
)

print('Days since last trade (excluding first-trade -1 sentinel):')
print(df[df['days_since_last_trade'] >= 0]['days_since_last_trade'].describe().round(1))
print()
print(f'First trades (sentinel -1): {(df["days_since_last_trade"] == -1).sum():,}')


Days since last trade (excluding first-trade -1 sentinel):
count    171012.0
mean         51.2
std         102.8
min           0.0
25%           1.0
50%           7.0
75%          51.0
max        1072.0
Name: days_since_last_trade, dtype: float64

First trades (sentinel -1): 35,797


### Cell 13 — Cluster signal: co-trading within a 5-day window

If multiple insiders at the *same company* all trade within 5 days of each other, that's a potential coordination signal.
One person buying is normal. Five people at the same firm all buying in the same week is suspicious.

**How it works:**
For each trade (company X, date D), we count how many *other* unique insiders at company X
made any trade between D−5 days and D+5 days.

**Note on performance:** This loops per company group. Most companies have <100 insider trades so it's fast.
For very large conglomerates it may take a few seconds.

In [26]:
!pip install tqdm
from tqdm.auto import tqdm   # progress bar — install with: pip install tqdm if missing

def compute_cluster_signal(df):
    results = []
    for (company,), grp in tqdm(df.groupby(['company']), desc='Computing cluster signal'):
        grp      = grp.sort_values('date_from').copy()
        dates    = grp['date_from'].values
        persons  = grp['person'].values
        counts   = np.zeros(len(grp), dtype=int)

        for i in range(len(grp)):
            d = dates[i]
            p = persons[i]
            window = np.timedelta64(5, 'D')
            in_window = (np.abs(dates - d) <= window) & (persons != p)
            counts[i] = len(set(persons[in_window]))

        grp['cluster_signal'] = counts
        results.append(grp)

    return pd.concat(results).sort_index()

df = compute_cluster_signal(df)

print('Cluster signal distribution (unique co-traders within ±5 days):')
print(df['cluster_signal'].value_counts().sort_index().head(10))
df.head(5)


You should consider upgrading via the '/Users/siddanthsatish/Desktop/ML predictor/.venv/bin/python3 -m pip install --upgrade pip' command.


Computing cluster signal: 100%|██████████| 2198/2198 [00:18<00:00, 117.25it/s]


Cluster signal distribution (unique co-traders within ±5 days):
cluster_signal
0    27009
1    18576
2    13148
3     9758
4     8110
5     6809
6     5694
7     5035
8     4485
9     4134
Name: count, dtype: int64


,company,person,category,buy_shares,buy_pct,buy_value,sale_shares,sale_pct,sale_value,pledge_shares,...,date_from,date_to,trade_year,trade_month,transaction_type,trade_value,trade_shares,trade_value_zscore,days_since_last_trade,cluster_signal
0,Pudumjee Paper Products Limited,3P LAND HOLDINGS LIMITED,Promoter,0.0,0.0,0.0,45900.0,0.0,2508435.0,0.0,...,2024-01-18,2024-01-18,2024,2024-01,sale_or_pledge_invoke,2508435.0,45900.0,0.000000,-1,1
1,360 ONE WAM LIMITED,5 PAISA CAPITAL LIMITED,Promoter,0.0,0.0,0.0,0.0,0.0,0.0,465.0,...,2023-01-20,2023-01-30,2023,2023-01,pledge_create_or_release,860177.0,465.0,-0.298299,-1,0
2,360 ONE WAM LIMITED,5 PAISA CAPITAL LIMITED,Promoter,0.0,0.0,0.0,182.0,0.0,77104.0,53147.0,...,2023-03-03,2023-03-23,2023,2023-03,mixed_sale_pledge,22867179.0,53329.0,2.572565,42,10
3,360 ONE WAM LIMITED,5 PAISA CAPITAL LIMITED,Promoter,0.0,0.0,0.0,17.0,0.0,7103.0,2735.0,...,2023-04-21,2023-05-30,2023,2023-04,mixed_sale_pledge,1127490.0,2752.0,-0.263428,49,1
4,360 ONE WAM LIMITED,5 PAISA CAPITAL LIMITED,Promoter,0.0,0.0,0.0,0.0,0.0,0.0,2210.0,...,2023-05-30,2023-06-13,2023,2023-05,pledge_create_or_release,930148.0,2210.0,-0.289171,39,10


### Cell 14 — Encode the `category` column

ML models need numbers, not text. The `category` column has 3 values:
- `Promoter` — founder/major shareholder, highest scrutiny
- `Designated Person` — senior employee with access to price-sensitive info
- `Promoter And Designated Person` — both roles

We use **one-hot encoding**: create one binary column per category.
"Promoter And Designated Person" gets both columns set to 1 — which is correct, they qualify for both.

We keep the original `category` column for human-readable output in Power BI.

In [27]:
df['is_promoter']          = df['category'].str.contains('Promoter',          case=False).astype(int)
df['is_designated_person'] = df['category'].str.contains('Designated Person', case=False).astype(int)

print('Category encoding check:')
print(df[['category', 'is_promoter', 'is_designated_person']].drop_duplicates().to_string(index=False))


Category encoding check:
                      category  is_promoter  is_designated_person
                      Promoter            1                     0
             Designated Person            0                     1
Promoter And Designated Person            1                     1


### Cell 15 — Final feature set preview and save

Let's see what we've built and save it as `features.csv`.
This file is the input to Step 3 (price data merge) and Step 4 (modelling).

In [28]:
FEATURE_COLS = [
    'company', 'person', 'category', 'transaction_type',
    'date_from', 'date_to', 'trade_year', 'trade_month',
    'trade_value', 'trade_shares', 'held_shares', 'held_pct',
    'trade_value_zscore', 'days_since_last_trade', 'cluster_signal',
    'is_promoter', 'is_designated_person',
]

df_features = df[FEATURE_COLS].copy()

print(f'Feature dataset shape: {df_features.shape[0]:,} rows × {df_features.shape[1]} columns')
print(f'\nNull counts:')
print(df_features.isnull().sum()[df_features.isnull().sum() > 0])

df_features.to_csv('features.csv', index=False)
print('\nSaved to features.csv')

df_features.head(5)

Feature dataset shape: 206,809 rows × 17 columns

Null counts:
Series([], dtype: int64)

Saved to features.csv


,company,person,category,transaction_type,date_from,date_to,trade_year,trade_month,trade_value,trade_shares,held_shares,held_pct,trade_value_zscore,days_since_last_trade,cluster_signal,is_promoter,is_designated_person
0,Pudumjee Paper Products Limited,3P LAND HOLDINGS LIMITED,Promoter,sale_or_pledge_invoke,2024-01-18,2024-01-18,2024,2024-01,2508435.0,45900.0,13567242.0,0.0,0.000000,-1,1,1,0
1,360 ONE WAM LIMITED,5 PAISA CAPITAL LIMITED,Promoter,pledge_create_or_release,2023-01-20,2023-01-30,2023,2023-01,860177.0,465.0,0.0,0.0,-0.298299,-1,0,1,0
2,360 ONE WAM LIMITED,5 PAISA CAPITAL LIMITED,Promoter,mixed_sale_pledge,2023-03-03,2023-03-23,2023,2023-03,22867179.0,53329.0,0.0,0.0,2.572565,42,10,1,0
3,360 ONE WAM LIMITED,5 PAISA CAPITAL LIMITED,Promoter,mixed_sale_pledge,2023-04-21,2023-05-30,2023,2023-04,1127490.0,2752.0,0.0,0.0,-0.263428,49,1,1,0
4,360 ONE WAM LIMITED,5 PAISA CAPITAL LIMITED,Promoter,pledge_create_or_release,2023-05-30,2023-06-13,2023,2023-05,930148.0,2210.0,0.0,0.0,-0.289171,39,10,1,0


---
## Step 4: Unsupervised Anomaly Detection (Modelling)

We have no "confirmed insider trading" labels — SEBI enforcement orders are rare and public.
So we use **unsupervised** models: they learn what a *normal* trade looks like from the bulk of the data,
then flag trades that don't fit that pattern as anomalies.

We'll run **three models** and compare what each flags:
1. **Isolation Forest** — randomly isolates points; anomalies are easier to isolate (fewer splits needed)
2. **One-Class SVM** — draws a tight boundary around the "normal" region; points outside = anomalies
3. **Autoencoder (PyTorch)** — a neural network trained to reconstruct its own input; trades it can't reconstruct well are anomalies

Each model produces an **anomaly score** per trade. Higher score = more suspicious.

### Cell 16 — Load features and prepare model input

We reload `features.csv` and select only the numeric columns the models will use.

Two pre-processing steps:
- **Log-transform** `trade_value` and `trade_shares`: these are heavily right-skewed
  (a few ₹100Cr trades dominate; most trades are small). Log squashes the scale so
  the model isn't overwhelmed by outliers on this one dimension.
- **Replace -1 sentinel** in `days_since_last_trade`: -1 means "first ever trade, no history."
  We replace it with the median gap so it doesn't mislead the model into thinking
  negative time has passed.

In [29]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('features.csv', low_memory=False)
df['date_from'] = pd.to_datetime(df['date_from'])

# Replace -1 sentinel with median gap before scaling
median_gap = df[df['days_since_last_trade'] >= 0]['days_since_last_trade'].median()
df['days_since_last_trade'] = df['days_since_last_trade'].replace(-1, median_gap)

# Log-transform skewed columns (log1p = log(1+x) handles zeros safely)
df['log_trade_value']  = np.log1p(df['trade_value'])
df['log_trade_shares'] = np.log1p(df['trade_shares'])

MODEL_FEATURES = [
    'log_trade_value',
    'log_trade_shares',
    'trade_value_zscore',
    'held_pct',
    'days_since_last_trade',
    'cluster_signal',
    'is_promoter',
    'is_designated_person',
]

X = df[MODEL_FEATURES].fillna(0).values

# Standardise: shift each feature to mean=0, std=1
# Required for SVM and Autoencoder; harmless for Isolation Forest
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'Model input shape: {X_scaled.shape}')
print(f'Features: {MODEL_FEATURES}')


Model input shape: (206809, 8)
Features: ['log_trade_value', 'log_trade_shares', 'trade_value_zscore', 'held_pct', 'days_since_last_trade', 'cluster_signal', 'is_promoter', 'is_designated_person']


### Cell 17 — Model 1: Isolation Forest

**Intuition:** Imagine randomly drawing lines to split the data.
Normal points are clustered together — you need many splits to isolate one.
Anomalous points are alone in sparse regions — just a few splits isolates them.
The model counts how many splits it takes. Fewer splits = more anomalous.

**Key parameters:**
- `n_estimators=200` — number of random trees (more = more stable scores)
- `contamination='auto'` — let sklearn decide the anomaly threshold internally

**Output:** `if_score` — higher value means more anomalous (we flip the sign of sklearn's raw output).

In [30]:
from sklearn.ensemble import IsolationForest

print('Training Isolation Forest...')
iso = IsolationForest(n_estimators=200, contamination='auto', random_state=42, n_jobs=-1)
iso.fit(X_scaled)

# decision_function: more negative = more anomalous → flip sign so higher = more suspicious
df['if_score'] = -iso.decision_function(X_scaled)

print('Done.')
print(f'\nIF score distribution:')
print(df['if_score'].describe().round(4))
print(f'\nTop 5 most suspicious trades (by Isolation Forest):')
df.nlargest(5, 'if_score')[['company', 'person', 'category', 'transaction_type',
                             'trade_value', 'if_score', 'date_from']].to_string(index=False)


Training Isolation Forest...
Done.

IF score distribution:
count    206809.0000
mean         -0.0843
std           0.0739
min          -0.1647
25%          -0.1431
50%          -0.1127
75%          -0.0377
max           0.2501
Name: if_score, dtype: float64

Top 5 most suspicious trades (by Isolation Forest):


'                         company                                person category transaction_type  trade_value  if_score  date_from\n       Berger Paints (I) Limited   U.K. Paints (India) Private Limited Promoter              buy 9.056328e+10  0.250134 2025-12-31\n          Univastu India Limited                    Pradeep Khandagale Promoter              buy 1.208471e+09  0.250069 2025-10-15\nNuvama Wealth Management Limited               PAGAC Ecstasy Pte. Ltd. Promoter              buy 9.242656e+10  0.250025 2025-12-27\n         Tega Industries Limited NIHAL FISCAL SERVICES PRIVATE LIMITED Promoter              buy 7.506951e+10  0.249328 2025-09-18\n       GPT Infraprojects Limited              GPT Sons Private Limited Promoter              buy 5.499222e+09  0.247566 2024-07-01'

### Cell 18 — Model 2: One-Class SVM (SGD approximation)

**Intuition:** The SVM draws the tightest possible boundary around the "normal" cloud of data points.
Anything that lands outside that boundary scores as an anomaly.

**Why SGD version?** The classic One-Class SVM is O(n²) — it would take hours on 210K rows.
`SGDOneClassSVM` is a fast linear approximation that trains in seconds and gives similar results.

**Output:** `ocsvm_score` — higher value means more anomalous.

In [34]:
from sklearn.linear_model import SGDOneClassSVM
from sklearn.pipeline import make_pipeline
from sklearn.kernel_approximation import Nystroem

print('Training One-Class SVM...')

# Nystroem approximates an RBF kernel, giving us non-linear boundaries
# without the O(n²) cost of a true kernel SVM
ocsvm_pipeline = make_pipeline(
    Nystroem(kernel='rbf', n_components=300, random_state=42),
    SGDOneClassSVM(nu=0.05, random_state=42)
)
ocsvm_pipeline.fit(X_scaled)

# decision_function: negative = anomaly → flip sign
df['ocsvm_score'] = -ocsvm_pipeline.decision_function(X_scaled)

print('Done.')
print(f'\nOCSVM score distribution:')
print(df['ocsvm_score'].describe().round(4))
print(f'\nTop 5 most suspicious trades (by One-Class SVM):')
df.nlargest(5, 'ocsvm_score')[['company', 'person', 'category', 'transaction_type',
                                'trade_value', 'ocsvm_score', 'date_from']].to_string(index=False)


Training One-Class SVM...
Done.

OCSVM score distribution:
count    206809.0000
mean         -0.0135
std           0.0100
min          -0.0407
25%          -0.0182
50%          -0.0156
75%          -0.0118
max           0.0780
Name: ocsvm_score, dtype: float64

Top 5 most suspicious trades (by One-Class SVM):


'                   company                                      person          category         transaction_type  trade_value  ocsvm_score  date_from\nAnand Rathi Wealth Limited ANAND RATHI SHARE AND STOCK BROKERS LIMITED          Promoter pledge_create_or_release 1686008975.0     0.078015 2024-05-30\n Sumeet Industries Limited                PADMINI POLYTEX PVT. LIMITED          Promoter                      buy 4388977500.0     0.078015 2025-07-22\n   Shriram Finance Limited       SHRIRAM INSIGHT SHARE BROKERS LIMITED          Promoter        mixed_sale_pledge  350248320.0     0.078015 2024-06-28\n Sumeet Industries Limited                                  JPB FIBRES          Promoter                      buy  947521100.0     0.078014 2025-07-22\n     Bajaj Finserv Limited                    BAJAJ FINSERV DIRECT LTD Designated Person    sale_or_pledge_invoke  996042515.0     0.078014 2025-11-18'

### Cell 19 — Model 3: Autoencoder (TensorFlow / Keras)

**Intuition:** An autoencoder is a neural network trained to compress its input down to a small
"bottleneck" and then reconstruct the original input from that compressed form.

```
Input (8 features) → [Encoder] → Bottleneck (2 numbers) → [Decoder] → Reconstructed input (8 features)
```

When trained on mostly-normal trades, it gets very good at reconstructing normal patterns.
When it sees an anomalous trade, it can't reconstruct it well — the **reconstruction error is high**.

That reconstruction error becomes our anomaly score.

Architecture: 8 → 4 → 2 → 4 → 8 (symmetric encoder/decoder)  
We use Keras (TensorFlow's high-level API) — you define layers, compile with a loss function, and call `.fit()`.

In [35]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import warnings
warnings.filterwarnings('ignore')

# ── Build the autoencoder ──────────────────────────────────────────────
input_dim = X_scaled.shape[1]

inputs  = keras.Input(shape=(input_dim,))
encoded = layers.Dense(4, activation='relu')(inputs)
encoded = layers.Dense(2, activation='relu')(encoded)   # bottleneck
decoded = layers.Dense(4, activation='relu')(encoded)
outputs = layers.Dense(input_dim, activation='linear')(decoded)

autoencoder = keras.Model(inputs, outputs, name='autoencoder')
autoencoder.compile(optimizer='adam', loss='mse')
autoencoder.summary()

# ── Train: input = output (reconstruct itself) ─────────────────────────
print('\nTraining Autoencoder...')
history = autoencoder.fit(
    X_scaled, X_scaled,
    epochs=25,
    batch_size=512,
    shuffle=True,
    verbose=1,
)

# ── Score: mean reconstruction error per row ───────────────────────────
reconstructed  = autoencoder.predict(X_scaled, verbose=0)
df['ae_score'] = np.mean((X_scaled - reconstructed) ** 2, axis=1)

print('\nAutoencoder score distribution:')
print(df['ae_score'].describe().round(6))
print('\nTop 5 most suspicious trades (by Autoencoder):')
print(df.nlargest(5, 'ae_score')[['company', 'person', 'category', 'transaction_type',
                                   'trade_value', 'ae_score', 'date_from']].to_string(index=False))

Model: "autoencoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 4)              │            36 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │            10 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 4)              │            12 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 8)              │            40 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 98 (392.00 B)

 Trainable params: 98 (392.00 B)

 Non-trainable params: 0 (0.00 B)


Training Autoencoder...
Epoch 1/25
404/404 ━━━━━━━━━━━━━━━━━━━━ 1s 772us/step - loss: 1.0416
Epoch 2/25
404/404 ━━━━━━━━━━━━━━━━━━━━ 0s 737us/step - loss: 0.7684
Epoch 3/25
404/404 ━━━━━━━━━━━━━━━━━━━━ 0s 733us/step - loss: 0.6322
Epoch 4/25
404/404 ━━━━━━━━━━━━━━━━━━━━ 0s 729us/step - loss: 0.5619
Epoch 5/25
404/404 ━━━━━━━━━━━━━━━━━━━━ 0s 728us/step - loss: 0.4951
Epoch 6/25
404/404 ━━━━━━━━━━━━━━━━━━━━ 0s 891us/step - loss: 0.4711
Epoch 7/25
404/404 ━━━━━━━━━━━━━━━━━━━━ 0s 731us/step - loss: 0.4648
Epoch 8/25
404/404 ━━━━━━━━━━━━━━━━━━━━ 0s 727us/step - loss: 0.4665
Epoch 9/25
404/404 ━━━━━━━━━━━━━━━━━━━━ 0s 732us/step - loss: 0.4750
Epoch 10/25
404/404 ━━━━━━━━━━━━━━━━━━━━ 0s 729us/step - loss: 0.4585
Epoch 11/25
404/404 ━━━━━━━━━━━━━━━━━━━━ 0s 727us/step - loss: 0.4595
Epoch 12/25
404/404 ━━━━━━━━━━━━━━━━━━━━ 0s 731us/step - loss: 0.4567
Epoch 13/25
404/404 ━━━━━━━━━━━━━━━━━━━━ 0s 744us/step - loss: 0.4484
Epoch 14/25
404/404 ━━━━━━━━━━━━━━━━━━━━ 0s 736us/step - loss: 0.4588
Epoc

### Cell 20 — Combine scores: ensemble anomaly score

Each model has its own scale (Isolation Forest scores are tiny decimals, Autoencoder scores
are mean squared errors). To compare them fairly we **normalise each score to a 0–1 range**,
then average them into a single `ensemble_score`.

A trade that all three models agree is suspicious will score close to 1.
A trade only one model flags might be a model-specific quirk rather than a real signal.

We also add a `models_agreed` column — how many of the 3 models put this trade in
their top 5% most suspicious. Trades where 2 or 3 models agree are the strongest candidates.

In [36]:
def normalise_01(series):
    mn, mx = series.min(), series.max()
    return (series - mn) / (mx - mn)

df['if_score_norm']    = normalise_01(df['if_score'])
df['ocsvm_score_norm'] = normalise_01(df['ocsvm_score'])
df['ae_score_norm']    = normalise_01(df['ae_score'])

df['ensemble_score'] = (
    df['if_score_norm'] +
    df['ocsvm_score_norm'] +
    df['ae_score_norm']
) / 3

# Flag top 5% of each model
threshold = 0.95
df['if_flag']    = (df['if_score_norm']    >= threshold).astype(int)
df['ocsvm_flag'] = (df['ocsvm_score_norm'] >= threshold).astype(int)
df['ae_flag']    = (df['ae_score_norm']    >= threshold).astype(int)
df['models_agreed'] = df['if_flag'] + df['ocsvm_flag'] + df['ae_flag']

print('Model agreement breakdown:')
print(df['models_agreed'].value_counts().sort_index().to_frame('trades'))
print()
print('Top 10 trades by ensemble score (all 3 models agreed):')
cols = ['company', 'person', 'category', 'transaction_type',
        'trade_value', 'ensemble_score', 'models_agreed', 'date_from']
df[df['models_agreed'] == 3].nlargest(10, 'ensemble_score')[cols].to_string(index=False)


Model agreement breakdown:
               trades
models_agreed        
0              206624
1                 168
2                  17

Top 10 trades by ensemble score (all 3 models agreed):


'Empty DataFrame\nColumns: [company, person, category, transaction_type, trade_value, ensemble_score, models_agreed, date_from]\nIndex: []'

### Cell 21 — Save results

We save two files:
- `anomaly_scores.csv` — the full dataset with all model scores attached (input for Power BI)
- `flagged_trades.csv` — only the trades where at least 2 models agreed (input for Step 5 price validation)

In [37]:
SCORE_COLS = [
    'company', 'person', 'category', 'transaction_type',
    'date_from', 'date_to', 'trade_year', 'trade_month',
    'trade_value', 'trade_shares', 'held_pct',
    'trade_value_zscore', 'days_since_last_trade', 'cluster_signal',
    'is_promoter', 'is_designated_person',
    'if_score', 'ocsvm_score', 'ae_score',
    'if_score_norm', 'ocsvm_score_norm', 'ae_score_norm',
    'ensemble_score', 'if_flag', 'ocsvm_flag', 'ae_flag', 'models_agreed',
]

df[SCORE_COLS].to_csv('anomaly_scores.csv', index=False)

flagged = df[df['models_agreed'] >= 2][SCORE_COLS].copy()
flagged.to_csv('flagged_trades.csv', index=False)

print(f'anomaly_scores.csv  → {len(df):,} rows (full dataset with scores)')
print(f'flagged_trades.csv  → {len(flagged):,} rows (2+ models agreed)')
print()
print('Sample flagged trades:')
flagged.nlargest(5, 'ensemble_score')[['company', 'person', 'transaction_type',
                                        'trade_value', 'ensemble_score', 'models_agreed', 'date_from']].to_string(index=False)


anomaly_scores.csv  → 206,809 rows (full dataset with scores)
flagged_trades.csv  → 17 rows (2+ models agreed)

Sample flagged trades:


'                  company                                person transaction_type  trade_value  ensemble_score  models_agreed  date_from\nSumeet Industries Limited          PADMINI POLYTEX PVT. LIMITED              buy 4.388978e+09        0.949873              2 2025-07-22\n  Tega Industries Limited NIHAL FISCAL SERVICES PRIVATE LIMITED              buy 7.506951e+10        0.669987              2 2025-09-18\nBerger Paints (I) Limited   U.K. Paints (India) Private Limited              buy 9.056328e+10        0.669769              2 2025-12-31\n   Univastu India Limited                    Pradeep Khandagale              buy 1.208471e+09        0.669159              2 2025-10-15\nGPT Infraprojects Limited              GPT Sons Private Limited              buy 5.499222e+09        0.664824              2 2024-07-01'

---
## Step 5: Price Validation for Flagged Trades

We have 17 trades flagged by at least 2 models. Now we ask: **did the stock price move significantly
after these trades?**

This is our proxy validation — since there are no confirmed "yes this was insider trading" labels,
we use subsequent price movement as a signal. A legitimate anomaly should precede an unusual price move.

**Important:** We are NOT using price data to train the models. We're using it AFTER scoring,
purely to validate whether the model's flags were meaningful. This avoids data leakage.

### Cell 22 — Load flagged trades and map company names to NSE tickers

yfinance needs ticker symbols like `BERGEPAINT.NS`, not names like `Berger Paints (I) Limited`.

We use yfinance's built-in search to find the best matching NSE ticker for each company name.
The search returns multiple candidates — we pick the first result with an `.NS` suffix (NSE India).

In [38]:
import pandas as pd
import yfinance as yf
import time

flagged = pd.read_csv('flagged_trades.csv')
flagged['date_from'] = pd.to_datetime(flagged['date_from'])
print(f'Flagged trades: {len(flagged)} across {flagged["company"].nunique()} companies')
print()

companies = flagged['company'].unique().tolist()

def find_nse_ticker(company_name):
    """Search yfinance for the best NSE ticker match for a company name."""
    try:
        results = yf.Search(company_name, max_results=5).quotes
        for r in results:
            symbol = r.get('symbol', '')
            if symbol.endswith('.NS'):
                return symbol
        return None
    except Exception:
        return None

print('Searching for NSE tickers...')
ticker_map = {}
for name in companies:
    ticker = find_nse_ticker(name)
    ticker_map[name] = ticker
    status = ticker if ticker else 'NOT FOUND'
    print(f'  {name[:55]:<55} → {status}')
    time.sleep(0.3)   # be polite to Yahoo Finance's rate limiter

found    = sum(1 for v in ticker_map.values() if v)
notfound = sum(1 for v in ticker_map.values() if not v)
print(f'\nFound: {found}  |  Not found: {notfound}')


Flagged trades: 17 across 17 companies

Searching for NSE tickers...
  Almondz Global Securities Limited                       → ALMONDZ.NS
  Bajaj Housing Finance Limited                           → NOT FOUND
  Syngene International Limited                           → SYNGENE.NS
  Capri Global Capital Limited                            → CGCL.NS
  Innovana Thinklabs Limited                              → NOT FOUND
  GPT Infraprojects Limited                               → GPTINFRA.NS
  Marine Electricals (India) Limited                      → MARINE.NS
  NRB Bearing Limited                                     → NRBBEARING.NS
  Tega Industries Limited                                 → TEGA.NS
  Sumeet Industries Limited                               → SUMEETINDS.NS
  Nuvama Wealth Management Limited                        → NUVAMA.NS
  PSP Projects Limited                                    → PSPPROJECT.NS
  IFCI Limited                                            → IFCI.NS
  Univastu 

### Cell 22b — Manual ticker overrides for unmatched companies

Two companies weren't found on NSE (`.NS`) by the auto-search but are listed on BSE (`.BO`).
BSE and NSE prices for the same stock are virtually identical, so using `.BO` tickers here is fine.
We simply patch them into `ticker_map` before pulling prices.

In [39]:
# Manual overrides for companies not found on NSE but available on BSE
MANUAL_TICKERS = {
    'Bajaj Housing Finance Limited': 'BAJAJHFL.BO',
    'Innovana Thinklabs Limited':    'INNOVANA.BO',
}

ticker_map.update(MANUAL_TICKERS)

print('Ticker map after manual overrides:')
for company, ticker in ticker_map.items():
    status = ticker if ticker else 'NOT FOUND'
    print(f'  {company[:55]:<55} → {status}')

found = sum(1 for v in ticker_map.values() if v)
print(f'\nTotal matched: {found}/{len(ticker_map)}')


Ticker map after manual overrides:
  Almondz Global Securities Limited                       → ALMONDZ.NS
  Bajaj Housing Finance Limited                           → BAJAJHFL.BO
  Syngene International Limited                           → SYNGENE.NS
  Capri Global Capital Limited                            → CGCL.NS
  Innovana Thinklabs Limited                              → INNOVANA.BO
  GPT Infraprojects Limited                               → GPTINFRA.NS
  Marine Electricals (India) Limited                      → MARINE.NS
  NRB Bearing Limited                                     → NRBBEARING.NS
  Tega Industries Limited                                 → TEGA.NS
  Sumeet Industries Limited                               → SUMEETINDS.NS
  Nuvama Wealth Management Limited                        → NUVAMA.NS
  PSP Projects Limited                                    → PSPPROJECT.NS
  IFCI Limited                                            → IFCI.NS
  Univastu India Limited                 

### Cell 23 — Pull historical price data from Yahoo Finance

For each company where we found a ticker, we download daily closing prices covering
the full period of our dataset plus 30 extra days (so we can compute 30-day forward returns
even for the most recent trades).

`yf.download()` fetches from Yahoo Finance — it needs an internet connection and
may take 10–20 seconds for 17 tickers.

In [40]:
from datetime import timedelta

# Only pull for companies where we found a ticker
valid_tickers = {name: t for name, t in ticker_map.items() if t}

if not valid_tickers:
    print('No tickers found — cannot pull price data.')
else:
    tickers_list = list(valid_tickers.values())
    start_date   = flagged['date_from'].min() - timedelta(days=5)
    end_date     = flagged['date_from'].max() + timedelta(days=45)

    print(f'Pulling price data for {len(tickers_list)} tickers from {start_date.date()} to {end_date.date()}...')

    raw_prices = yf.download(
        tickers_list,
        start=start_date,
        end=end_date,
        auto_adjust=True,
        progress=True,
    )

    # Extract just closing prices, flatten multi-level columns
    if isinstance(raw_prices.columns, pd.MultiIndex):
        prices = raw_prices['Close']
    else:
        prices = raw_prices[['Close']].rename(columns={'Close': tickers_list[0]})

    print(f'\nPrice data shape: {prices.shape}')
    print(f'Nulls per ticker (missing trading days are normal):')
    print(prices.isnull().sum().sort_values(ascending=False).head(10))


Pulling price data for 17 tickers from 2024-03-14 to 2026-02-14...


[*********************100%***********************]  17 of 17 completed


Price data shape: (476, 17)
Nulls per ticker (missing trading days are normal):
Ticker
INNOVANA.BO      188
BAJAJHFL.BO      125
PSPPROJECT.NS      1
SUMEETINDS.NS      1
ALMONDZ.NS         0
UNIVASTU.NS        0
TEGA.NS            0
SYNGENE.NS         0
SFL.NS             0
NRBBEARING.NS      0
dtype: int64


### Cell 24 — Compute forward price returns (5 / 10 / 30 days after each trade)

For each flagged trade we look up the stock price on the trade date and again 5, 10,
and 30 trading days later. The percentage change is the **forward return**.

A large absolute forward return (up OR down) after a trade is a sign the insider
may have traded on information about an upcoming event.

We use `abs()` because insider trading can precede both good news (buy before announcement)
and bad news (sell before a crash).

In [41]:
def get_forward_return(ticker, trade_date, days_ahead, price_df):
    """Return % price change from trade_date to trade_date + days_ahead trading days."""
    if ticker not in price_df.columns:
        return None
    col = price_df[ticker].dropna()
    if col.empty:
        return None

    # Find the nearest available trading day on or after trade_date
    future_dates = col.index[col.index >= trade_date]
    if len(future_dates) == 0:
        return None
    start_price_date = future_dates[0]
    start_price      = col.loc[start_price_date]

    # Find price days_ahead trading days later
    idx = col.index.get_loc(start_price_date)
    end_idx = idx + days_ahead
    if end_idx >= len(col):
        return None
    end_price = col.iloc[end_idx]

    return round((end_price - start_price) / start_price * 100, 2)

# Add ticker column to flagged dataframe
flagged['ticker'] = flagged['company'].map(ticker_map)

for days in [5, 10, 30]:
    col_name = f'fwd_return_{days}d'
    flagged[col_name] = flagged.apply(
        lambda row: get_forward_return(row['ticker'], row['date_from'], days, prices)
        if pd.notna(row['ticker']) else None,
        axis=1
    )

print('Forward returns added. Preview:')
return_cols = ['company', 'person', 'transaction_type', 'trade_value',
               'ensemble_score', 'fwd_return_5d', 'fwd_return_10d', 'fwd_return_30d']
print(flagged[return_cols].to_string(index=False))


Forward returns added. Preview:
                           company                                           person      transaction_type  trade_value  ensemble_score  fwd_return_5d  fwd_return_10d  fwd_return_30d
 Almondz Global Securities Limited AVONMORE CAPITAL AND MANAGEMENT SERVICES LIMITED sale_or_pledge_invoke 1.818776e+08        0.655378          -2.99           17.57           44.09
     Bajaj Housing Finance Limited                            BAJAJ FINANCE LIMITED     mixed_sale_pledge 1.587832e+10        0.655819           1.72            3.17           -1.77
     Syngene International Limited                                   BIOCON LIMITED sale_or_pledge_invoke 6.861450e+09        0.653717          -2.51           -2.54           -5.11
      Capri Global Capital Limited            CAPRI GLOBAL HOLDINGS PRIVATE LIMITED                   buy 4.217821e+10        0.641198          -0.61            7.48            3.47
        Innovana Thinklabs Limited                        

### Cell 25 — Save validated flagged trades

We save the flagged trades with forward returns attached as `flagged_validated.csv`.
This becomes the "interesting cases" table in the Power BI dashboard.

In [42]:
flagged.to_csv('flagged_validated.csv', index=False)
print(f'Saved flagged_validated.csv — {len(flagged)} rows')
print()
print('Trades with largest absolute 30-day price move after the trade:')
flagged['abs_30d'] = flagged['fwd_return_30d'].abs()
print(
    flagged.nlargest(5, 'abs_30d')[
        ['company', 'person', 'transaction_type', 'ensemble_score',
         'fwd_return_5d', 'fwd_return_10d', 'fwd_return_30d']
    ].to_string(index=False)
)


Saved flagged_validated.csv — 17 rows

Trades with largest absolute 30-day price move after the trade:
                           company                                           person      transaction_type  ensemble_score  fwd_return_5d  fwd_return_10d  fwd_return_30d
Marine Electricals (India) Limited                          KDU Enterprises Pvt Ltd                   buy        0.642936          20.29           53.51           74.90
 Almondz Global Securities Limited AVONMORE CAPITAL AND MANAGEMENT SERVICES LIMITED sale_or_pledge_invoke        0.655378          -2.99           17.57           44.09
                      IFCI Limited                               PRESIDENT OF INDIA sale_or_pledge_invoke        0.642413           7.45           20.21           30.32
         GPT Infraprojects Limited                         GPT Sons Private Limited                   buy        0.664824          15.59           33.38           24.67
        Whirlpool of India Limited                  

---
## Step 6: Export for Power BI

Power BI can read CSV files directly. We'll also create an SQLite database as an
alternative — useful if the CSV files get large or you want relational queries in Power BI.

We export three tables:
1. `anomaly_scores.csv` — all 210K trades with model scores (already saved in Step 4)
2. `flagged_validated.csv` — the 17 flagged trades with price validation (just saved)
3. `company_summary.csv` — one row per company: average score, number of flagged trades, top insider

These three tables connect in Power BI via the `company` column.

### Cell 26 — Build company-level summary table

This table gives Power BI a single row per company with aggregated stats —
useful for the heatmap and ranking visuals.

In [43]:
import pandas as pd
import sqlite3

scores = pd.read_csv('anomaly_scores.csv')
scores['date_from'] = pd.to_datetime(scores['date_from'])

company_summary = (
    scores.groupby('company')
    .agg(
        total_trades      = ('ensemble_score', 'count'),
        avg_ensemble_score= ('ensemble_score', 'mean'),
        max_ensemble_score= ('ensemble_score', 'max'),
        flagged_count     = ('models_agreed',  lambda x: (x >= 2).sum()),
        unique_insiders   = ('person',         'nunique'),
        total_trade_value = ('trade_value',    'sum'),
        latest_trade_date = ('date_from',      'max'),
    )
    .reset_index()
    .sort_values('max_ensemble_score', ascending=False)
)

company_summary.to_csv('company_summary.csv', index=False)
print(f'Saved company_summary.csv — {len(company_summary):,} companies')
print()
print('Top 10 companies by max anomaly score:')
print(company_summary.head(10)[
    ['company', 'total_trades', 'flagged_count', 'avg_ensemble_score', 'max_ensemble_score']
].to_string(index=False))


Saved company_summary.csv — 2,198 companies

Top 10 companies by max anomaly score:
                         company  total_trades  flagged_count  avg_ensemble_score  max_ensemble_score
       Sumeet Industries Limited            31              1            0.291426            0.949873
         Tega Industries Limited            90              1            0.167667            0.669987
       Berger Paints (I) Limited            53              1            0.164706            0.669769
          Univastu India Limited            14              1            0.336072            0.669159
       GPT Infraprojects Limited            58              1            0.267608            0.664824
      Innovana Thinklabs Limited             8              1            0.385372            0.663774
Nuvama Wealth Management Limited          1111              1            0.126500            0.662994
            PSP Projects Limited            11              1            0.213162            0.65878

### Cell 27 — Export to SQLite database

SQLite is a single-file database — no server needed. Power BI can connect to it directly.
This is optional but useful if you want to demo SQL queries over the data.

In [44]:
import sqlite3

conn = sqlite3.connect('insider_trading.db')

scores.to_sql('anomaly_scores',   conn, if_exists='replace', index=False)
flagged.to_sql('flagged_trades',  conn, if_exists='replace', index=False)
company_summary.to_sql('company_summary', conn, if_exists='replace', index=False)

conn.close()

print('Saved insider_trading.db with 3 tables:')
print('  anomaly_scores   — full dataset with model scores')
print('  flagged_trades   — flagged trades with price validation')
print('  company_summary  — one row per company')
print()

# Quick verification
conn = sqlite3.connect('insider_trading.db')
for table in ['anomaly_scores', 'flagged_trades', 'company_summary']:
    count = pd.read_sql(f'SELECT COUNT(*) as n FROM {table}', conn).iloc[0,0]
    print(f'  {table:<25} {count:>8,} rows')
conn.close()


Saved insider_trading.db with 3 tables:
  anomaly_scores   — full dataset with model scores
  flagged_trades   — flagged trades with price validation
  company_summary  — one row per company

  anomaly_scores             206,809 rows
  flagged_trades                  17 rows
  company_summary              2,198 rows


---
## Step 7: Export for Power BI Web

Power BI web (app.powerbi.com) works best with an Excel file.
We'll export all three tables into one `.xlsx` file with separate sheets —
one upload, everything connected.

### Cell 28 — Export to Excel for Power BI

Three sheets in one file:
- **AllTrades** — all 210K trades with anomaly scores (for timeline and heatmap visuals)
- **FlaggedTrades** — the 17 flagged trades with forward price returns (for the investigation table)
- **CompanySummary** — one row per company with aggregated stats (for ranking and heatmap)

In [45]:
import pandas as pd

scores  = pd.read_csv('anomaly_scores.csv')
flagged = pd.read_csv('flagged_validated.csv')
summary = pd.read_csv('company_summary.csv')

# Convert dates to string so Excel doesn't mangle them
for df in [scores, flagged]:
    for col in ['date_from', 'date_to']:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col]).dt.strftime('%Y-%m-%d')

if 'latest_trade_date' in summary.columns:
    summary['latest_trade_date'] = pd.to_datetime(summary['latest_trade_date']).dt.strftime('%Y-%m-%d')

output_path = 'insider_trading_powerbi.xlsx'

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    scores.to_excel( writer, sheet_name='AllTrades',     index=False)
    flagged.to_excel(writer, sheet_name='FlaggedTrades', index=False)
    summary.to_excel(writer, sheet_name='CompanySummary',index=False)

print(f'Saved: {output_path}')
print(f'  Sheet AllTrades:      {len(scores):,} rows')
print(f'  Sheet FlaggedTrades:  {len(flagged):,} rows')
print(f'  Sheet CompanySummary: {len(summary):,} rows')
print()
print('Upload this file to app.powerbi.com to build the dashboard.')


Saved: insider_trading_powerbi.xlsx
  Sheet AllTrades:      206,809 rows
  Sheet FlaggedTrades:  17 rows
  Sheet CompanySummary: 2,198 rows

Upload this file to app.powerbi.com to build the dashboard.
